# Step 9 — Final clustering (weather regime definition)


## Domains

The notebook is organised in two parts: **Europe** (`REGION = "europe"`, cells below the introduction through the first save and evaluation block), then **North America** (`REGION = "northamerica"`, second half). Helper functions in the second code cell are shared.

## Purpose
Define discrete **weather regimes** by clustering daily atmospheric states in EOF/PCA space.

## Inputs
- `PCs_reduced(time, mode)` from Step 7 (`pipeline_prepro.py` / `preprocessing_helpers.py`; originally prototyped in the now-retired `07_PCA.ipynb`)
- `X_matrix(time, space)` from Step 4–6 preprocessing output (for regime composites in physical space)

## Method
- **K-means**: hard assignment (each day belongs to exactly one regime)

## Outputs
- `regime_labels(time)`
- centroids in PC space, regime frequencies, persistence, transition matrix
- regime composites in lat–lon space

## Role in workflow
Turns continuous EOF variability (Step 7) into discrete, recurring dynamical states for interpretation and downstream analysis.


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from common.map_plotting_helpers import add_contextily_basemap_under_spatial_raster as _add_map_basemap

from sklearn.cluster import KMeans

plt.rcParams.update({"figure.dpi": 120})

import os

from common.preprocessing_helpers import (
    bbox_step_filename_suffix,
    filter_by_bbox_step as _filter_by_bbox_step,
    filter_by_sigma_mode as _filter_by_sigma_mode,
    filter_paths_by_yeo_johnson,
    find_module_root,
    pick_single as _pick_single,
    yeo_johnson_enabled,
)

# Notebook lives in src/; headless nbconvert may run from src/_headless/ — not Path("..").
module_root = find_module_root()

# ------------------------
# Configuration
# ------------------------

REGION = os.environ.get("REGION", "europe")  # "europe" | "northamerica"
BBOX_STEP: str | None = os.environ.get("BBOX_STEP", "B4")  # e.g. "B0".."B5"
BBOX_STEP = (str(BBOX_STEP).strip().upper() if BBOX_STEP else None)
SIGMA_MODE = str(os.environ.get("SIGMA_MODE", "")).strip().lower()

BBOX_OUT_TAG = bbox_step_filename_suffix(module_root, REGION, BBOX_STEP)

# Disambiguates Step 9/10-11 outputs from pre-YJ runs at the same k/w/eps/var/bbox
# (Step 7/8 already carry this via the `opt` label baked into their filenames).
YJ_TAG = "_yj" if yeo_johnson_enabled() else ""

preprocessed_dir = module_root / "data" / "preprocessed" / REGION
processed_dir = module_root / "data" / "processed" / REGION
results_clustering_dir = module_root / "results" / "clustering" / REGION

YEAR_START = int(os.environ.get("YEAR_START", "1979"))
YEAR_END = int(os.environ.get("YEAR_END", "2025"))
YEAR_SPAN = os.environ.get("YEAR_SPAN", "") or (f"{YEAR_START}-{YEAR_END}" if YEAR_START != YEAR_END else f"{YEAR_START}")

# Must match Steps 4–8
WINDOW_DAYS = int(os.environ.get("WINDOW_DAYS", "11"))
EPSILON_M = float(os.environ.get("EPSILON_M", "20.0"))
VAR_THRESHOLD = float(os.environ.get("VAR_THRESHOLD", "0.90"))

# Choose final number of regimes (from Step 8 diagnostics)
FINAL_K = int(os.environ.get("FINAL_K", "5"))

# Reproducibility
RANDOM_STATE = int(os.environ.get("RANDOM_STATE", "42"))


# Stage-1 Z500 clustering method for this run:
#   "ward"        — pure Ward hierarchical clustering (Step-8 linkage cut at FINAL_K, no K-means at all)
#   "ward+kmeans" — Ward-seeded K-means (Step-8 centroids as K-means init; previous default/hybrid
#                   behaviour — still a K-means fit_predict, just no longer randomly initialised)
def _normalize_cluster_mode(value: str) -> str:
    v = str(value).strip().lower().replace("-", "").replace("_", "")
    if v in ("ward", "w", "hrc", "wardonly"):
        return "ward"
    if v in ("ward+kmeans", "wardkmeans", "wardkm", "hybrid", "wk"):
        return "ward_kmeans"
    raise ValueError(f"CLUSTER_MODE must be 'ward' or 'ward+kmeans', got {value!r}")


CLUSTER_MODE = _normalize_cluster_mode(os.environ.get("CLUSTER_MODE", "ward"))
# On-disk method tag: pure Ward gets its own "ward" filename family; the Ward-seeded K-means
# hybrid keeps the existing "kmeans" tag (it IS a K-means result, just seeded differently), so
# previously-computed hybrid runs stay valid without recomputation.
_METHOD_TAG = "ward" if CLUSTER_MODE == "ward" else "kmeans"
print(f"[step9] CLUSTER_MODE={CLUSTER_MODE!r} -> output method tag {_METHOD_TAG!r}")


def _hint_step7_pca_var_mismatch(
    processed_dir: Path,
    *,
    window_days: int,
    eps: int,
    year_span: str,
    bbox_step: str | None,
    requested_var: int,
) -> str:
    """If varNN in the filename does not match, list which var tags exist on disk for same w/eps/year/bbox."""
    broad = sorted(processed_dir.glob(f"07_Z500_PCA_w{window_days:02d}_eps{eps:02d}_var*_{year_span}.nc"))
    if bbox_step:
        step = str(bbox_step).strip().upper()
        broad = [p for p in broad if f"_step{step}_" in p.name or f"step{step}_" in p.name]
    broad = _filter_by_sigma_mode(broad, SIGMA_MODE, kind="Step 7 PCA")
    broad = filter_paths_by_yeo_johnson(broad)
    tags: set[int] = set()
    for p in broad:
        for part in p.stem.split("_"):
            if part.startswith("var") and len(part) > 3 and part[3:].isdecimal():
                tags.add(int(part[3:]))
                break
    if not tags:
        return f"No 07_Z500_PCA_*_var*_{year_span}.nc under {processed_dir} for this w/eps/bbox."
    other = sorted(tags - {requested_var})
    if not other:
        return ""
    pairs = ", ".join(f"{t/100:.2f} (--var-threshold) → var{t:02d} in filename" for t in other)
    return f"You asked for var{requested_var:02d} but on disk for this bbox+window+eps+year: {pairs}."


def resolve_step7_pca(
    processed_dir: Path, window_days: int, epsilon_m: float, var_threshold: float, *, year_span: str, bbox_step: str | None
) -> Path:
    eps = int(round(epsilon_m))
    var = int(var_threshold * 100)
    pats = [
        f"07_Z500_PCA_w{window_days:02d}_eps{eps:02d}_var{var:02d}_*_{year_span}.nc",
        f"07_Z500_PCA_w{window_days:02d}_eps{eps:02d}_var{var:02d}_{year_span}.nc",
        f"07_Z500_PCA_w{window_days:02d}_eps{eps:02d}_var{var:02d}.nc",
    ]
    cands: list[Path] = []
    for pat in pats:
        cands.extend(sorted(processed_dir.glob(pat)))
    if not cands:
        hint = _hint_step7_pca_var_mismatch(
            processed_dir,
            window_days=window_days,
            eps=eps,
            year_span=year_span,
            bbox_step=bbox_step,
            requested_var=var,
        )
        raise FileNotFoundError(
            f"No Step 7 PCA file for var{var:02d} (VAR_THRESHOLD={var_threshold}), "
            f"w={window_days}, eps={eps}, years={year_span}. {hint}"
        )
    cands = _filter_by_bbox_step(cands, bbox_step, kind="Step 7 PCA")
    cands = _filter_by_sigma_mode(cands, SIGMA_MODE, kind="Step 7 PCA")
    cands = filter_paths_by_yeo_johnson(cands)
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in cands:
        if p not in seen and p.exists():
            seen.add(p)
            unique.append(p)
    return _pick_single(unique, kind=f"Step 7 PCA (REGION={REGION}, w={window_days}, eps={eps}, var={var}, years={year_span}, bbox_step={bbox_step}, sigma_mode={SIGMA_MODE})")


def resolve_step4_6_preproc(
    preprocessed_dir: Path, window_days: int, epsilon_m: float, *, year_span: str, bbox_step: str | None
) -> Path:
    eps_label = int(round(float(epsilon_m)))
    pats = [
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_*_{year_span}.nc",
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_{year_span}.nc",
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}.nc",
    ]
    cands: list[Path] = []
    for pat in pats:
        cands.extend(sorted(preprocessed_dir.glob(pat)))
    if not cands:
        raise FileNotFoundError(
            f"No Step 4–6 preprocessed files for w={window_days}, eps={eps_label}, years={year_span} under {preprocessed_dir}. "
            "Check WINDOW_DAYS, EPSILON_M, YEAR_SPAN, and that preprocessing was run for this bbox."
        )
    cands = _filter_by_bbox_step(cands, bbox_step, kind="Step 4–6 preprocessed")
    cands = _filter_by_sigma_mode(cands, SIGMA_MODE, kind="Step 4–6 preprocessed")
    cands = filter_paths_by_yeo_johnson(cands)
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in cands:
        if p not in seen and p.exists():
            seen.add(p)
            unique.append(p)
    return _pick_single(unique, kind=f"Step 4–6 preprocessed (REGION={REGION}, w={window_days}, eps={eps_label}, years={year_span}, bbox_step={bbox_step}, sigma_mode={SIGMA_MODE})")


paths = {
    "in_pca": resolve_step7_pca(processed_dir, WINDOW_DAYS, EPSILON_M, VAR_THRESHOLD, year_span=YEAR_SPAN, bbox_step=BBOX_STEP),
    "in_preproc": resolve_step4_6_preproc(preprocessed_dir, WINDOW_DAYS, EPSILON_M, year_span=YEAR_SPAN, bbox_step=BBOX_STEP),
    "out_regimes": results_clustering_dir
    / f"09_Z500_regimes_{_METHOD_TAG}_k{FINAL_K:02d}_w{WINDOW_DAYS:02d}_eps{int(round(EPSILON_M)):02d}_var{int(VAR_THRESHOLD*100):02d}{YJ_TAG}_{YEAR_SPAN}{BBOX_OUT_TAG}.nc",
}

paths


In [ ]:
# Figures shared with the CLI PDF (`summary_reports` → `regime_plotting`): use these builders
# so notebook output matches the summary PDF for the same NetCDF.
from common.step08_09_figures import (
    build_figure_pc12_kmeans,
    build_figure_pc12_ward,
    build_figure_regime_frequency_bar,
    build_figure_step9_regime_frequency_from_ds,
)
from common.step1011_figures import build_figure_regime_field_panels_cell12


In [ ]:
def persistence_by_regime(labels: np.ndarray, n_regimes: int) -> dict[int, dict[str, float]]:
    """Mean/median run length per regime. Labels are 1..n_regimes."""
    labels = np.asarray(labels).astype(int)
    out: dict[int, dict[str, float]] = {}

    for r in range(1, n_regimes + 1):
        rl = []
        i = 0
        while i < labels.size:
            if labels[i] == r:
                j = i
                while j < labels.size and labels[j] == r:
                    j += 1
                rl.append(j - i)
                i = j
            else:
                i += 1

        if len(rl) == 0:
            out[r] = {"mean": float("nan"), "median": float("nan"), "n_runs": 0}
        else:
            arr = np.asarray(rl, dtype=int)
            out[r] = {"mean": float(np.mean(arr)), "median": float(np.median(arr)), "n_runs": int(arr.size)}

    return out


def transition_matrix(labels: np.ndarray, n_regimes: int) -> np.ndarray:
    """Count transitions i->j between consecutive days. Labels are 1..n_regimes."""
    labels = np.asarray(labels).astype(int)
    T = np.zeros((n_regimes, n_regimes), dtype=int)
    for a, b in zip(labels[:-1], labels[1:]):
        if 1 <= a <= n_regimes and 1 <= b <= n_regimes:
            T[a - 1, b - 1] += 1
    return T


def plot_transition_matrix(T: np.ndarray, title: str) -> None:
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(T, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("to regime")
    ax.set_ylabel("from regime")
    ax.set_xticks(np.arange(T.shape[1]))
    ax.set_yticks(np.arange(T.shape[0]))
    ax.set_xticklabels(np.arange(1, T.shape[1] + 1))
    ax.set_yticklabels(np.arange(1, T.shape[0] + 1))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for i in range(T.shape[0]):
        for j in range(T.shape[1]):
            ax.text(
                j,
                i,
                str(T[i, j]),
                ha="center",
                va="center",
                color="white" if T[i, j] > T.max() / 2 else "black",
            )

    plt.tight_layout()
    plt.show()

In [ ]:
# Load PCA outputs (PC scores)
ds_pca = xr.open_dataset(paths["in_pca"], chunks=None)
if "PCs_reduced" not in ds_pca:
    raise KeyError(f"Expected 'PCs_reduced' in {paths['in_pca']}, found: {list(ds_pca.data_vars)}")

PC_da = ds_pca["PCs_reduced"].load()
time_dim = PC_da.dims[0]
mode_dim = PC_da.dims[1]
time_coord = PC_da[time_dim] if time_dim in PC_da.coords else ds_pca[time_dim]

X_pca = np.asarray(PC_da.values, dtype=np.float32)
assert float(np.isfinite(X_pca).mean()) == 1.0

# Load preprocessing output for composites in physical space
# We composite the *actual* weighted anomaly fields (not just centroid reconstructions).
ds_pre = xr.open_dataset(paths["in_preproc"], chunks=None)
if "X_matrix" not in ds_pre:
    raise KeyError(f"Expected 'X_matrix' in {paths['in_preproc']}, found: {list(ds_pre.data_vars)}")

Xmat_da = ds_pre["X_matrix"].load()
X_matrix = np.asarray(Xmat_da.values, dtype=np.float32)
assert X_matrix.shape[0] == X_pca.shape[0]
assert float(np.isfinite(X_matrix).mean()) == 1.0

# Get lat/lon for reshaping composites
if "z500_weighted" in ds_pre:
    z_w = ds_pre["z500_weighted"]
    lat_dim = "lat" if "lat" in z_w.dims else "latitude"
    lon_dim = "lon" if "lon" in z_w.dims else "longitude"
    lat = z_w[lat_dim].values
    lon = z_w[lon_dim].values
else:
    lat = ds_pre.get("lat", ds_pre.get("latitude"))
    lon = ds_pre.get("lon", ds_pre.get("longitude"))
    if lat is None or lon is None:
        raise KeyError("Could not infer lat/lon from preprocessing dataset.")
    lat = np.asarray(lat)
    lon = np.asarray(lon)

n_lat = int(len(lat))
n_lon = int(len(lon))

print("X_pca:", X_pca.shape)
print("X_matrix:", X_matrix.shape)
print("lat/lon:", n_lat, n_lon)

ds_pca

In [ ]:
# Stage-1 clustering: pure Ward HRC, or Ward-seeded K-means (CLUSTER_MODE, set in the settings cell).
from scipy.cluster.hierarchy import fcluster

_ward_hc_path = os.environ.get("IN_STEP8_HC", "").strip()
_Z_ward = None
if _ward_hc_path:
    _Z_ward = xr.open_dataset(_ward_hc_path)["ward_linkage"].values

if CLUSTER_MODE == "ward":
    if _Z_ward is None:
        raise FileNotFoundError(
            "CLUSTER_MODE='ward' requires IN_STEP8_HC (a Step-8 08_Z500_HC_Ward_*.nc) to be set and "
            "readable. run_regime_clustering.py sets this automatically whenever the Step-8 "
            "file exists for this bbox/region; set it manually (env var) if running this notebook "
            "standalone."
        )
    kmeans = None
    labels0 = fcluster(_Z_ward, FINAL_K, criterion="maxclust") - 1  # 0..k-1, matches K-means convention
    regime_labels = labels0.astype(int) + 1  # 1..k
    centroids = np.stack(
        [X_pca[regime_labels == r].mean(axis=0) for r in range(1, FINAL_K + 1)], axis=0
    ).astype(np.float32)
    print(f"[step9] Pure Ward HRC labels (k={FINAL_K}) from {_ward_hc_path}")
else:
    _ward_init = None
    if _Z_ward is not None:
        _ward_asgn = fcluster(_Z_ward, FINAL_K, criterion="maxclust")  # 1-indexed
        _ward_init = np.stack([X_pca[_ward_asgn == c].mean(axis=0) for c in range(1, FINAL_K + 1)], axis=0)
        print(f"[step9] Ward→k-means init from {_ward_hc_path}")
    else:
        print("[step9] IN_STEP8_HC not set; K-means will use random init (no Ward seeding).")

    kmeans = (
        KMeans(n_clusters=FINAL_K, init=_ward_init, n_init=1, random_state=RANDOM_STATE)
        if _ward_init is not None
        else KMeans(n_clusters=FINAL_K, n_init=50, random_state=RANDOM_STATE)
    )
    labels0 = kmeans.fit_predict(X_pca)  # 0..k-1
    regime_labels = labels0.astype(int) + 1  # 1..k
    centroids = kmeans.cluster_centers_.astype(np.float32)

print("regime_labels unique:", np.unique(regime_labels))
print("centroids shape:", centroids.shape)

# PC-space quick look (PC1 vs PC2) — same builders as the PDF Step 9 extras
pc1 = X_pca[:, 0]
pc2 = X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros_like(pc1)

if CLUSTER_MODE == "ward":
    _fig_pc = build_figure_pc12_ward(pc1, pc2, regime_labels, k=FINAL_K, fig_num=None)
else:
    _fig_pc = build_figure_pc12_kmeans(pc1, pc2, regime_labels, k=FINAL_K, fig_num=None)
plt.show()


In [ ]:
# Elbow plot (inertia) for K-means, k=1..11
# Note: inertia is the within-cluster sum of squares (SSE); lower is better.

ks = list(range(1, 12))
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE)
    km.fit(X_pca)
    inertias.append(float(km.inertia_))

plt.figure(figsize=(6, 3))
plt.plot(ks, inertias, marker="o")
plt.title("Elbow plot: K-means inertia vs k")
plt.xlabel("k")
plt.ylabel("inertia (SSE)")
plt.xticks(ks)
plt.grid(True, alpha=0.3)
plt.show()

list(zip(ks, inertias))[:5], list(zip(ks, inertias))[-5:]

In [ ]:
# Regime frequency, persistence, transitions
_method_label = "Ward HRC" if CLUSTER_MODE == "ward" else "K-means"

counts = np.bincount(regime_labels, minlength=FINAL_K + 1)[1:]
frac = counts / counts.sum()

print("counts:", counts)
print("fractions:", np.round(frac, 3))

_regc = np.arange(1, FINAL_K + 1, dtype=int)
_fig_freq = build_figure_regime_frequency_bar(
    _regc, frac.astype(float), k=FINAL_K, ward=(CLUSTER_MODE == "ward"), fig_num=None
)
plt.show()

pers = persistence_by_regime(regime_labels, FINAL_K)
pers

T = transition_matrix(regime_labels, FINAL_K)
plot_transition_matrix(T, title=f"Transition counts ({_method_label}, k={FINAL_K})")

# Normalised transition probabilities (row-normalised)
row_sum = T.sum(axis=1, keepdims=True)
P = np.where(row_sum > 0, T / row_sum, np.nan)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(P, cmap="magma", vmin=0, vmax=np.nanmax(P))
ax.set_title(f"Transition probabilities ({_method_label}, k={FINAL_K})")
ax.set_xlabel("to regime")
ax.set_ylabel("from regime")
ax.set_xticks(np.arange(FINAL_K))
ax.set_yticks(np.arange(FINAL_K))
ax.set_xticklabels(np.arange(1, FINAL_K + 1))
ax.set_yticklabels(np.arange(1, FINAL_K + 1))
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Regime composites in physical space (using the *actual* weighted fields)
# X_matrix is (time, space) with deterministic (lat outer, lon inner) order.

X_maps = X_matrix.reshape((X_matrix.shape[0], n_lat, n_lon))

composites = np.zeros((FINAL_K, n_lat, n_lon), dtype=np.float32)

for r in range(1, FINAL_K + 1):
    idx = np.where(regime_labels == r)[0]
    if idx.size == 0:
        composites[r - 1, :, :] = np.nan
        continue
    composites[r - 1, :, :] = np.mean(X_maps[idx, :, :], axis=0)

# Plot composites — same panel layout as PDF / Step 9 `composite_weighted` figure
_da_comp = xr.DataArray(
    composites,
    dims=("regime", "latitude", "longitude"),
    coords={
        "regime": np.arange(1, FINAL_K + 1, dtype=int),
        "latitude": lat,
        "longitude": lon,
    },
    name="composite_weighted",
    attrs={"long_name": "regime composite (mean field)", "units": ""},
)
_fig_maps = build_figure_regime_field_panels_cell12(
    _da_comp,
    suptitle=f"Regime composites (mean fields) — {_method_label}, k={FINAL_K}",
    fig_num=None,
)
plt.show()


In [ ]:
# Save regimes to NetCDF
out = xr.Dataset(attrs=dict(ds_pca.attrs))

out["regime_labels"] = xr.DataArray(
    regime_labels.astype(np.int16),
    dims=[time_dim],
    coords={time_dim: time_coord},
)

out["centroids_pc"] = xr.DataArray(
    centroids.astype(np.float32),
    dims=["regime", mode_dim],
    coords={"regime": np.arange(1, FINAL_K + 1), mode_dim: PC_da[mode_dim].values},
)

out["frequency"] = xr.DataArray(
    frac.astype(np.float32),
    dims=["regime"],
    coords={"regime": np.arange(1, FINAL_K + 1)},
)

out["transition_counts"] = xr.DataArray(
    T.astype(np.int32),
    dims=["regime_from", "regime_to"],
    coords={
        "regime_from": np.arange(1, FINAL_K + 1),
        "regime_to": np.arange(1, FINAL_K + 1),
    },
)

out["transition_prob"] = xr.DataArray(
    P.astype(np.float32),
    dims=["regime_from", "regime_to"],
    coords={
        "regime_from": np.arange(1, FINAL_K + 1),
        "regime_to": np.arange(1, FINAL_K + 1),
    },
)

out["composite_weighted"] = xr.DataArray(
    composites.astype(np.float32),
    dims=["regime", lat_dim, lon_dim],
    coords={"regime": np.arange(1, FINAL_K + 1), lat_dim: lat, lon_dim: lon},
)

# Persistence summary as attributes (compact)
for r in range(1, FINAL_K + 1):
    out.attrs[f"persistence_mean_regime_{r}"] = float(pers[r]["mean"])
    out.attrs[f"persistence_median_regime_{r}"] = float(pers[r]["median"])
    out.attrs[f"persistence_n_runs_regime_{r}"] = int(pers[r]["n_runs"])

out.attrs.update(
    {
        "step": "09_final_clustering_regimes",
        "method": _METHOD_TAG,         # "ward" or "kmeans" (on-disk filename family)
        "cluster_mode": CLUSTER_MODE,   # "ward" or "ward_kmeans" (exact algorithm actually used)
        "final_k": int(FINAL_K),
        "random_state": int(RANDOM_STATE),
        "source_pca": str(paths["in_pca"]),
        "source_preproc": str(paths["in_preproc"]),
        "source_step8_hc": _ward_hc_path or "",
        "note": "Regime composites are computed from the actual weighted anomaly fields (X_matrix) averaged over days in each regime.",
    }
)

# Atomic write: write tmp then replace (avoids locked-file overwrite issues)
import os
from pathlib import Path

paths["out_regimes"].parent.mkdir(parents=True, exist_ok=True)
_target = Path(paths["out_regimes"])
_tmp = _target.with_suffix(_target.suffix + ".tmp")

try:
    out.to_netcdf(_tmp)
    os.replace(_tmp, _target)
    saved_path = _target
except PermissionError:
    # If overwrite is denied (file lock / sync), write a copy instead.
    if _tmp.exists():
        try:
            _tmp.unlink()
        except Exception:
            pass
    saved_path = _target.with_name(_target.stem + "_copy" + _target.suffix)
    out.to_netcdf(saved_path)

saved_path


## Evaluation of the final regime clustering (K-means)

This section adds **per-regime** quality diagnostics and writes them to the same NetCDF output file.

> Note: physical-space pattern correlation is often the easiest to explain in a paper, because it ties the cluster to the actual Z500 anomaly maps.


In [ ]:
# Setup (robust: can reload labels from saved NetCDF)
from pathlib import Path

import numpy as np
import xarray as xr
from sklearn.metrics import silhouette_samples, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import pairwise_distances

# ---- Load regime labels / centroids if not in memory ----
if "regime_labels" not in globals():
    ds_reg = None

    # 1) If an output Dataset exists in memory, use it
    if "out" in globals() and isinstance(out, xr.Dataset) and ("regime_labels" in out):
        ds_reg = out

    # 2) Else try to open the saved NetCDF (preferred)
    elif "paths" in globals() and isinstance(paths, dict) and ("out_regimes" in paths):
        p = Path(paths["out_regimes"])  # may be PosixPath already
        if p.exists():
            # Open, read needed arrays, then close immediately to avoid file locks.
            with xr.open_dataset(p) as _ds:
                regime_labels = _ds["regime_labels"].values
                if "centroids" not in globals() and ("centroids_pc" in _ds):
                    centroids = _ds["centroids_pc"].values
            ds_reg = "closed"  # sentinel

    if ds_reg is None:
        raise NameError(
            "`regime_labels` is not defined. Run the clustering cells above first, "
            "or ensure `paths['out_regimes']` points to an existing NetCDF produced by this notebook."
        )

    if "regime_labels" not in globals():
        # when ds_reg came from `out` in memory
        regime_labels = ds_reg["regime_labels"].values
        if "centroids" not in globals() and ("centroids_pc" in ds_reg):
            centroids = ds_reg["centroids_pc"].values

# ---- Core inputs (must exist for the chosen metrics) ----
labels = np.asarray(regime_labels, dtype=int)  # 1..K

if "FINAL_K" in globals():
    K = int(FINAL_K)
else:
    K = int(np.nanmax(labels))

if "X_pca" not in globals():
    raise NameError("`X_pca` is not defined. Run the PCA/clustering prep cells above before evaluating.")

if "centroids" not in globals():
    raise NameError("`centroids` is not defined. Run the clustering cell above (Ward or K-means), or load a NetCDF that includes `centroids_pc`.")

centroids_pc = np.asarray(centroids, dtype=np.float64)
X_pc = np.asarray(X_pca, dtype=np.float64)



### 1) Cohesion (how tight is regime \(r\)?) — computed in **PC/EOF space**
K-means explicitly minimizes within-cluster squared distance in PC space, so cohesion is most consistent there.

- **Distance to centroid** (mean/median/p90/p95): lower means members of regime \(r\) are closer to its centroid → more compact regime.
  - **Interpretation**: these values are scale-dependent (depend on the number of PCs and their scaling), so use them mainly to compare regimes *within the same run*.
  - **Rule of thumb**: regimes with very large p95 (relative to others) often contain mixed states / outliers.

- **Silhouette (per regime)**: ranges from \(-1\) to \(1\). Higher is better.
  - **\(>0.5\)**: well separated / cohesive
  - **\(0.25–0.5\)**: some overlap but still useful
  - **\(<0.25\)**: substantial overlap
  - **negative**: many days are closer to another regime than to their assigned one




In [ ]:
# 1) Cohesion: distance-to-centroid summaries + per-regime silhouette (PC space)
dist_to_centroid = np.full(X_pc.shape[0], np.nan, dtype=np.float64)
for r in range(1, K + 1):
    idx = np.where(labels == r)[0]
    if idx.size == 0:
        continue
    c = centroids_pc[r - 1]
    dist_to_centroid[idx] = np.linalg.norm(X_pc[idx] - c[None, :], axis=1)

cohesion_mean = np.full(K, np.nan)
cohesion_median = np.full(K, np.nan)
cohesion_p90 = np.full(K, np.nan)
cohesion_p95 = np.full(K, np.nan)
cohesion_radius_typical = np.full(K, np.nan)  # robust within-regime radius

for r in range(1, K + 1):
    d = dist_to_centroid[labels == r]
    if d.size == 0:
        continue
    cohesion_mean[r - 1] = float(np.mean(d))
    cohesion_median[r - 1] = float(np.median(d))
    cohesion_p90[r - 1] = float(np.quantile(d, 0.90))
    cohesion_p95[r - 1] = float(np.quantile(d, 0.95))
    cohesion_radius_typical[r - 1] = float(np.median(d))

sil_samples = silhouette_samples(X_pc, labels, metric="euclidean")
sil_mean_by_regime = np.full(K, np.nan)
for r in range(1, K + 1):
    s = sil_samples[labels == r]
    if s.size == 0:
        continue
    sil_mean_by_regime[r - 1] = float(np.mean(s))

import pandas as pd
cohesion_table = pd.DataFrame(
    {
        "regime": np.arange(1, K + 1),
        "cohesion_dist_mean_pc": cohesion_mean,
        "cohesion_dist_median_pc": cohesion_median,
        "cohesion_dist_p90_pc": cohesion_p90,
        "cohesion_dist_p95_pc": cohesion_p95,
        "silhouette_mean_pc": sil_mean_by_regime,
    }
)
cohesion_table

### 2) Separation (how distinct is regime \(r\) from others?) — centroids in **PC space**
- **Nearest-centroid distance**: distance from centroid \(c_r\) to the closest other centroid. Higher means more distinct.
- **Separation-to-spread ratio**: \(\text{nearest-centroid distance} / \text{typical within-regime radius}\).
  - **\(> 1\)**: the regime’s centroid is farther from its nearest neighbor than its typical radius → usually distinct.
  - **\(< 1\)**: overlap likely (members can be as close to the neighbor as to their own centroid).




In [ ]:
# 2) Separation: nearest-centroid distance + separation/spread ratio (PC space)
Dcc = pairwise_distances(centroids_pc, metric="euclidean")
np.fill_diagonal(Dcc, np.inf)
nearest_centroid_dist = Dcc.min(axis=1).astype(np.float64)
separation_to_spread = nearest_centroid_dist / cohesion_radius_typical

separation_table = pd.DataFrame(
    {
        "regime": np.arange(1, K + 1),
        "nearest_centroid_dist_pc": nearest_centroid_dist,
        "separation_to_spread_pc": separation_to_spread,
    }
)
separation_table

### 3) Stability / robustness (is regime \(r\) reproducible?) — bootstrap resampling
We rerun K-means on bootstrap samples and compare the resulting full-sample assignments to the original.
Because cluster ids are arbitrary, we first **match bootstrap centroids to original centroids (Hungarian algorithm)**.

- **ARI (Adjusted Rand Index)**: 0 means random agreement, 1 means identical partitions.
- **NMI (Normalized Mutual Information)**: 0 to 1, higher is more agreement.
  - **\(\ge 0.8\)**: very stable
  - **\(0.6–0.8\)**: reasonably stable
  - **\(< 0.6\)**: sensitive to sampling (often means too many regimes, weak separation, or strong seasonality mixing)



In [ ]:
# 3) Stability / robustness: bootstrap resampling + Hungarian matching, then ARI/NMI
# Mode-aware: refits the same clustering method used for the official labels (CLUSTER_MODE) on each
# bootstrap resample, so the ARI/NMI comparison is apples-to-apples (Ward-vs-Ward or K-means-vs-K-means).
# If scipy is unavailable in the environment, this cell will fall back to NaNs.
try:
    from scipy.cluster.hierarchy import linkage as _linkage
    from scipy.optimize import linear_sum_assignment

    N_BOOT = 30
    rng = np.random.default_rng(RANDOM_STATE)
    ari_boot = np.empty(N_BOOT, dtype=np.float64)
    nmi_boot = np.empty(N_BOOT, dtype=np.float64)

    for b in range(N_BOOT):
        boot_idx = rng.integers(0, X_pc.shape[0], size=X_pc.shape[0])

        if CLUSTER_MODE == "ward":
            _Zb = _linkage(X_pc[boot_idx], method="ward")
            _asgn_b = fcluster(_Zb, K, criterion="maxclust")
            Cb = np.stack(
                [X_pc[boot_idx][_asgn_b == c].mean(axis=0) for c in range(1, K + 1)], axis=0
            )
            labels_b = np.argmin(pairwise_distances(X_pc, Cb), axis=1).astype(int) + 1
        else:
            km_b = KMeans(
                n_clusters=K,
                n_init=50,
                random_state=int(rng.integers(0, 1_000_000_000)),
            )
            km_b.fit(X_pc[boot_idx])
            labels_b = km_b.predict(X_pc).astype(int) + 1  # 1..K
            Cb = np.asarray(km_b.cluster_centers_, dtype=np.float64)

        # Match bootstrap regime ids to original regime ids by centroid distances
        cost = pairwise_distances(centroids_pc, Cb, metric="euclidean")
        row_ind, col_ind = linear_sum_assignment(cost)
        mapping = {int(col_ind[i] + 1): int(row_ind[i] + 1) for i in range(K)}
        labels_b_matched = np.vectorize(mapping.get)(labels_b)

        ari_boot[b] = adjusted_rand_score(labels, labels_b_matched)
        nmi_boot[b] = normalized_mutual_info_score(labels, labels_b_matched)

    stability_ari_mean = float(np.mean(ari_boot))
    stability_ari_std = float(np.std(ari_boot))
    stability_nmi_mean = float(np.mean(nmi_boot))
    stability_nmi_std = float(np.std(nmi_boot))
    stability_error = None

except Exception as e:
    N_BOOT = 30
    stability_ari_mean = np.nan
    stability_ari_std = np.nan
    stability_nmi_mean = np.nan
    stability_nmi_std = np.nan
    stability_error = str(e)

pd.DataFrame(
    {
        "metric": ["ARI", "NMI"],
        "mean": [stability_ari_mean, stability_nmi_mean],
        "std": [stability_ari_std, stability_nmi_std],
        "n_boot": [N_BOOT, N_BOOT],
    }
)


### 4) Physical-space “pattern purity” (do members look like the composite?)
Using the anomaly fields \(X\_\text{matrix}(time, space)\) and the regime composite maps:

- **Pattern correlation (per day vs its regime composite)**: reported as median and IQR (p25–p75) per regime.
  - **Interpretation** (very rough guidance):
    - **\(>0.6\)**: strong, consistent physical pattern
    - **\(0.3–0.6\)**: moderate consistency
    - **\(<0.3\)**: weak / mixed pattern

In [ ]:
# 4) Physical-space pattern purity: corr(day anomaly map, regime composite map)
# (Uses the same anomaly fields used to compute composites.)
if "X_maps" not in globals():
    n_lat, n_lon = len(lat), len(lon)
    X_maps = X_matrix.reshape((X_matrix.shape[0], n_lat, n_lon))

composites_maps = np.asarray(composites, dtype=np.float64)  # (K, lat, lon)

patterncorr_median = np.full(K, np.nan)
patterncorr_p25 = np.full(K, np.nan)
patterncorr_p75 = np.full(K, np.nan)

for r in range(1, K + 1):
    idx = np.where(labels == r)[0]
    if idx.size == 0:
        continue

    y = composites_maps[r - 1].ravel()
    y0 = y - y.mean()
    y_norm = np.linalg.norm(y0)
    if y_norm == 0:
        continue

    cors = np.empty(idx.size, dtype=np.float64)
    for j, t in enumerate(idx):
        x = np.asarray(X_maps[t], dtype=np.float64).ravel()
        x0 = x - x.mean()
        x_norm = np.linalg.norm(x0)
        cors[j] = np.nan if x_norm == 0 else float(np.dot(x0, y0) / (x_norm * y_norm))

    patterncorr_median[r - 1] = float(np.nanmedian(cors))
    patterncorr_p25[r - 1] = float(np.nanquantile(cors, 0.25))
    patterncorr_p75[r - 1] = float(np.nanquantile(cors, 0.75))

patterncorr_table = pd.DataFrame(
    {
        "regime": np.arange(1, K + 1),
        "patterncorr_median": patterncorr_median,
        "patterncorr_p25": patterncorr_p25,
        "patterncorr_p75": patterncorr_p75,
    }
)
patterncorr_table

In [ ]:
# Save evaluation metrics into the output Dataset and overwrite NetCDF
regime_coord = np.arange(1, K + 1)

out["cohesion_dist_mean_pc"] = xr.DataArray(
    cohesion_mean.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["cohesion_dist_median_pc"] = xr.DataArray(
    cohesion_median.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["cohesion_dist_p90_pc"] = xr.DataArray(
    cohesion_p90.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["cohesion_dist_p95_pc"] = xr.DataArray(
    cohesion_p95.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["silhouette_mean_pc"] = xr.DataArray(
    sil_mean_by_regime.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)

out["nearest_centroid_dist_pc"] = xr.DataArray(
    nearest_centroid_dist.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["separation_to_spread_pc"] = xr.DataArray(
    separation_to_spread.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)

out["patterncorr_median_composite"] = xr.DataArray(
    patterncorr_median.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["patterncorr_p25_composite"] = xr.DataArray(
    patterncorr_p25.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["patterncorr_p75_composite"] = xr.DataArray(
    patterncorr_p75.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)

out.attrs["stability_bootstrap_n"] = int(N_BOOT)
out.attrs["stability_bootstrap_ARI_mean"] = float(stability_ari_mean)
out.attrs["stability_bootstrap_ARI_std"] = float(stability_ari_std)
out.attrs["stability_bootstrap_NMI_mean"] = float(stability_nmi_mean)
out.attrs["stability_bootstrap_NMI_std"] = float(stability_nmi_std)
if stability_error is not None:
    out.attrs["stability_bootstrap_error"] = stability_error

# Atomic overwrite (same rationale as above)
import os
from pathlib import Path

_target = Path(paths["out_regimes"])
_tmp = _target.with_suffix(_target.suffix + ".tmp")

try:
    out.to_netcdf(_tmp)
    os.replace(_tmp, _target)
    saved_path = _target
except PermissionError:
    if _tmp.exists():
        try:
            _tmp.unlink()
        except Exception:
            pass
    saved_path = _target.with_name(_target.stem + "_copy" + _target.suffix)
    out.to_netcdf(saved_path)

print("Saved evaluation metrics to:", saved_path)


## North America

Same pipeline as above for **`REGION = "northamerica"`**. Run these cells after the Europe section (shared helper definitions in the second notebook cell still apply).


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from common.map_plotting_helpers import add_contextily_basemap_under_spatial_raster as _add_map_basemap

from sklearn.cluster import KMeans

plt.rcParams.update({"figure.dpi": 120})

import os

from common.preprocessing_helpers import find_module_root

module_root = find_module_root() if "module_root" not in globals() else module_root

# ------------------------
# Configuration
# ------------------------

REGION = "northamerica"  # duplicate block (same as Europe)
preprocessed_dir = module_root / "data" / "preprocessed" / REGION
processed_dir = module_root / "data" / "processed" / REGION
results_clustering_dir = module_root / "results" / "clustering" / REGION

YEAR_START = 1979
YEAR_END = 2025
YEAR_SPAN = f"{YEAR_START}-{YEAR_END}" if YEAR_START != YEAR_END else f"{YEAR_START}"

# Must match Steps 4–8
WINDOW_DAYS = 11
EPSILON_M = 20.0
VAR_THRESHOLD = 0.90

# Choose final number of regimes (from Step 8 diagnostics)
FINAL_K = 4

# Reproducibility
RANDOM_STATE = 42


# Same CLUSTER_MODE convention as the Europe block above ("ward" | "ward+kmeans").
def _normalize_cluster_mode(value: str) -> str:
    v = str(value).strip().lower().replace("-", "").replace("_", "")
    if v in ("ward", "w", "hrc", "wardonly"):
        return "ward"
    if v in ("ward+kmeans", "wardkmeans", "wardkm", "hybrid", "wk"):
        return "ward_kmeans"
    raise ValueError(f"CLUSTER_MODE must be 'ward' or 'ward+kmeans', got {value!r}")


CLUSTER_MODE = _normalize_cluster_mode(os.environ.get("CLUSTER_MODE", "ward"))
_METHOD_TAG = "ward" if CLUSTER_MODE == "ward" else "kmeans"
print(f"[step9] CLUSTER_MODE={CLUSTER_MODE!r} -> output method tag {_METHOD_TAG!r}")


def resolve_step7_pca(processed_dir: Path, window_days: int, epsilon_m: float, var_threshold: float, *, year_span: str) -> Path:
    expected = processed_dir / f"07_Z500_PCA_w{window_days:02d}_eps{int(round(epsilon_m)):02d}_var{int(var_threshold*100):02d}_{year_span}.nc"
    if expected.exists():
        return expected
    legacy = processed_dir / f"07_Z500_PCA_w{window_days:02d}_eps{int(round(epsilon_m)):02d}_var{int(var_threshold*100):02d}.nc"
    if legacy.exists():
        return legacy
    available = sorted(x.name for x in processed_dir.glob("07_Z500_PCA_w*_eps*_var*.nc"))
    raise FileNotFoundError(
        f"Could not find Step 7 PCA file for year_span={year_span!r} in {processed_dir}. "
        f"Expected {expected.name}. Available: {available}"
    )


def resolve_step4_6_preproc(preprocessed_dir: Path, window_days: int, epsilon_m: float, *, year_span: str) -> Path:
    eps_label = int(round(float(epsilon_m)))
    expected = preprocessed_dir / f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_{year_span}.nc"
    if expected.exists():
        return expected
    legacy = preprocessed_dir / f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}.nc"
    if legacy.exists():
        return legacy
    available = sorted(x.name for x in preprocessed_dir.glob("04_05_06_Z500_preprocessed_w*_eps*.nc"))
    raise FileNotFoundError(
        f"Could not find Step 4–6 preprocessed file for year_span={year_span!r} in {preprocessed_dir}. "
        f"Expected {expected.name}. Available: {available}"
    )


# Drop objects from the Europe block so PCA/clustering state is not mixed across domains
for _name in (
    "ds_pca",
    "ds_pre",
    "PC_da",
    "Xmat_da",
    "time_dim",
    "mode_dim",
    "time_coord",
    "X_pca",
    "X_matrix",
    "lat",
    "lon",
    "lat_dim",
    "lon_dim",
    "n_lat",
    "n_lon",
    "kmeans",
    "labels0",
    "regime_labels",
    "centroids",
    "composites",
    "X_maps",
    "counts",
    "frac",
    "pers",
    "T",
    "P",
    "out",
    "centroids_pc",
    "X_pc",
    "labels",
):
    globals().pop(_name, None)

paths = {
    "in_pca": resolve_step7_pca(processed_dir, WINDOW_DAYS, EPSILON_M, VAR_THRESHOLD, year_span=YEAR_SPAN),
    "in_preproc": resolve_step4_6_preproc(preprocessed_dir, WINDOW_DAYS, EPSILON_M, year_span=YEAR_SPAN),
    "out_regimes": results_clustering_dir
    / f"09_Z500_regimes_{_METHOD_TAG}_k{FINAL_K:02d}_w{WINDOW_DAYS:02d}_eps{int(round(EPSILON_M)):02d}_var{int(VAR_THRESHOLD*100):02d}_{YEAR_SPAN}.nc",
}

paths


In [ ]:
# Load PCA outputs (PC scores)
ds_pca = xr.open_dataset(paths["in_pca"], chunks=None)
if "PCs_reduced" not in ds_pca:
    raise KeyError(f"Expected 'PCs_reduced' in {paths['in_pca']}, found: {list(ds_pca.data_vars)}")

PC_da = ds_pca["PCs_reduced"].load()
time_dim = PC_da.dims[0]
mode_dim = PC_da.dims[1]
time_coord = PC_da[time_dim] if time_dim in PC_da.coords else ds_pca[time_dim]

X_pca = np.asarray(PC_da.values, dtype=np.float32)
assert float(np.isfinite(X_pca).mean()) == 1.0

# Load preprocessing output for composites in physical space
# We composite the *actual* weighted anomaly fields (not just centroid reconstructions).
ds_pre = xr.open_dataset(paths["in_preproc"], chunks=None)
if "X_matrix" not in ds_pre:
    raise KeyError(f"Expected 'X_matrix' in {paths['in_preproc']}, found: {list(ds_pre.data_vars)}")

Xmat_da = ds_pre["X_matrix"].load()
X_matrix = np.asarray(Xmat_da.values, dtype=np.float32)
assert X_matrix.shape[0] == X_pca.shape[0]
assert float(np.isfinite(X_matrix).mean()) == 1.0

# Get lat/lon for reshaping composites
if "z500_weighted" in ds_pre:
    z_w = ds_pre["z500_weighted"]
    lat_dim = "lat" if "lat" in z_w.dims else "latitude"
    lon_dim = "lon" if "lon" in z_w.dims else "longitude"
    lat = z_w[lat_dim].values
    lon = z_w[lon_dim].values
else:
    lat = ds_pre.get("lat", ds_pre.get("latitude"))
    lon = ds_pre.get("lon", ds_pre.get("longitude"))
    if lat is None or lon is None:
        raise KeyError("Could not infer lat/lon from preprocessing dataset.")
    lat = np.asarray(lat)
    lon = np.asarray(lon)

n_lat = int(len(lat))
n_lon = int(len(lon))

print("X_pca:", X_pca.shape)
print("X_matrix:", X_matrix.shape)
print("lat/lon:", n_lat, n_lon)

ds_pca

In [ ]:
# Stage-1 clustering: pure Ward HRC, or Ward-seeded K-means (CLUSTER_MODE, set above).
from scipy.cluster.hierarchy import fcluster

_ward_hc_path = os.environ.get("IN_STEP8_HC", "").strip()
_Z_ward = None
if _ward_hc_path:
    _Z_ward = xr.open_dataset(_ward_hc_path)["ward_linkage"].values

if CLUSTER_MODE == "ward":
    if _Z_ward is None:
        raise FileNotFoundError(
            "CLUSTER_MODE='ward' requires IN_STEP8_HC (a Step-8 08_Z500_HC_Ward_*.nc) to be set and "
            "readable — set it manually (env var) before running this block standalone."
        )
    kmeans = None
    labels0 = fcluster(_Z_ward, FINAL_K, criterion="maxclust") - 1  # 0..k-1, matches K-means convention
    regime_labels = labels0.astype(int) + 1  # 1..k
    centroids = np.stack(
        [X_pca[regime_labels == r].mean(axis=0) for r in range(1, FINAL_K + 1)], axis=0
    ).astype(np.float32)
    print(f"[step9] Pure Ward HRC labels (k={FINAL_K}) from {_ward_hc_path}")
else:
    _ward_init = None
    if _Z_ward is not None:
        _ward_asgn = fcluster(_Z_ward, FINAL_K, criterion="maxclust")  # 1-indexed
        _ward_init = np.stack([X_pca[_ward_asgn == c].mean(axis=0) for c in range(1, FINAL_K + 1)], axis=0)
        print(f"[step9] Ward→k-means init from {_ward_hc_path}")
    else:
        print("[step9] IN_STEP8_HC not set; K-means will use random init (no Ward seeding).")

    kmeans = (
        KMeans(n_clusters=FINAL_K, init=_ward_init, n_init=1, random_state=RANDOM_STATE)
        if _ward_init is not None
        else KMeans(n_clusters=FINAL_K, n_init=50, random_state=RANDOM_STATE)
    )
    labels0 = kmeans.fit_predict(X_pca)  # 0..k-1
    regime_labels = labels0.astype(int) + 1  # 1..k
    centroids = kmeans.cluster_centers_.astype(np.float32)

print("regime_labels unique:", np.unique(regime_labels))
print("centroids shape:", centroids.shape)

# PC-space quick look (PC1 vs PC2) — same builders as the PDF Step 9 extras
pc1 = X_pca[:, 0]
pc2 = X_pca[:, 1] if X_pca.shape[1] > 1 else np.zeros_like(pc1)

if CLUSTER_MODE == "ward":
    _fig_pc = build_figure_pc12_ward(pc1, pc2, regime_labels, k=FINAL_K, fig_num=None)
else:
    _fig_pc = build_figure_pc12_kmeans(pc1, pc2, regime_labels, k=FINAL_K, fig_num=None)
plt.show()


In [ ]:
# Elbow plot (inertia) for K-means, k=1..11
# Note: inertia is the within-cluster sum of squares (SSE); lower is better.

ks = list(range(1, 12))
inertias = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=50, random_state=RANDOM_STATE)
    km.fit(X_pca)
    inertias.append(float(km.inertia_))

plt.figure(figsize=(6, 3))
plt.plot(ks, inertias, marker="o")
plt.title("Elbow plot: K-means inertia vs k")
plt.xlabel("k")
plt.ylabel("inertia (SSE)")
plt.xticks(ks)
plt.grid(True, alpha=0.3)
plt.show()

list(zip(ks, inertias))[:5], list(zip(ks, inertias))[-5:]

In [ ]:
# Regime frequency, persistence, transitions

# Make this cell runnable even if helper-definition cells were skipped.
if "persistence_by_regime" not in globals():

    def persistence_by_regime(labels: np.ndarray, n_regimes: int) -> dict[int, dict[str, float]]:
        """Mean/median run length per regime. Labels are 1..n_regimes."""
        labels = np.asarray(labels).astype(int)
        out: dict[int, dict[str, float]] = {}

        for r in range(1, n_regimes + 1):
            rl = []
            i = 0
            while i < labels.size:
                if labels[i] == r:
                    j = i
                    while j < labels.size and labels[j] == r:
                        j += 1
                    rl.append(j - i)
                    i = j
                else:
                    i += 1

            if len(rl) == 0:
                out[r] = {"mean": float("nan"), "median": float("nan"), "n_runs": 0}
            else:
                arr = np.asarray(rl, dtype=int)
                out[r] = {"mean": float(np.mean(arr)), "median": float(np.median(arr)), "n_runs": int(arr.size)}

        return out


if "transition_matrix" not in globals():

    def transition_matrix(labels: np.ndarray, n_regimes: int) -> np.ndarray:
        """Count transitions i->j between consecutive days. Labels are 1..n_regimes."""
        labels = np.asarray(labels).astype(int)
        T = np.zeros((n_regimes, n_regimes), dtype=int)
        for a, b in zip(labels[:-1], labels[1:]):
            if 1 <= a <= n_regimes and 1 <= b <= n_regimes:
                T[a - 1, b - 1] += 1
        return T


if "plot_transition_matrix" not in globals():

    def plot_transition_matrix(T: np.ndarray, title: str) -> None:
        fig, ax = plt.subplots(figsize=(4.5, 4))
        im = ax.imshow(T, cmap="viridis")
        ax.set_title(title)
        ax.set_xlabel("to regime")
        ax.set_ylabel("from regime")
        ax.set_xticks(np.arange(T.shape[1]))
        ax.set_yticks(np.arange(T.shape[0]))
        ax.set_xticklabels(np.arange(1, T.shape[1] + 1))
        ax.set_yticklabels(np.arange(1, T.shape[0] + 1))
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        for i in range(T.shape[0]):
            for j in range(T.shape[1]):
                ax.text(
                    j,
                    i,
                    str(T[i, j]),
                    ha="center",
                    va="center",
                    color="white" if T[i, j] > T.max() / 2 else "black",
                )

        plt.tight_layout()
        plt.show()


_method_label = "Ward HRC" if CLUSTER_MODE == "ward" else "K-means"

counts = np.bincount(regime_labels, minlength=FINAL_K + 1)[1:]
frac = counts / counts.sum()

print("counts:", counts)
print("fractions:", np.round(frac, 3))

_regc = np.arange(1, FINAL_K + 1, dtype=int)
_fig_freq = build_figure_regime_frequency_bar(
    _regc, frac.astype(float), k=FINAL_K, ward=(CLUSTER_MODE == "ward"), fig_num=None
)
plt.show()

pers = persistence_by_regime(regime_labels, FINAL_K)
pers

T = transition_matrix(regime_labels, FINAL_K)
plot_transition_matrix(T, title=f"Transition counts ({_method_label}, k={FINAL_K})")

# Normalised transition probabilities (row-normalised)
row_sum = T.sum(axis=1, keepdims=True)
P = np.where(row_sum > 0, T / row_sum, np.nan)

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(P, cmap="magma", vmin=0, vmax=np.nanmax(P))
ax.set_title(f"Transition probabilities ({_method_label}, k={FINAL_K})")
ax.set_xlabel("to regime")
ax.set_ylabel("from regime")
ax.set_xticks(np.arange(FINAL_K))
ax.set_yticks(np.arange(FINAL_K))
ax.set_xticklabels(np.arange(1, FINAL_K + 1))
ax.set_yticklabels(np.arange(1, FINAL_K + 1))
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Regime composites in physical space (using the *actual* weighted fields)
# X_matrix is (time, space) with deterministic (lat outer, lon inner) order.

X_maps = X_matrix.reshape((X_matrix.shape[0], n_lat, n_lon))

composites = np.zeros((FINAL_K, n_lat, n_lon), dtype=np.float32)

for r in range(1, FINAL_K + 1):
    idx = np.where(regime_labels == r)[0]
    if idx.size == 0:
        composites[r - 1, :, :] = np.nan
        continue
    composites[r - 1, :, :] = np.mean(X_maps[idx, :, :], axis=0)

# Plot composites — same panel layout as PDF / Step 9 `composite_weighted` figure
_da_comp = xr.DataArray(
    composites,
    dims=("regime", "latitude", "longitude"),
    coords={
        "regime": np.arange(1, FINAL_K + 1, dtype=int),
        "latitude": lat,
        "longitude": lon,
    },
    name="composite_weighted",
    attrs={"long_name": "regime composite (mean field)", "units": ""},
)
_fig_maps = build_figure_regime_field_panels_cell12(
    _da_comp,
    suptitle=f"Regime composites (mean fields) — {_method_label}, k={FINAL_K}",
    fig_num=None,
)
plt.show()


In [ ]:
# Save regimes to NetCDF
out = xr.Dataset(attrs=dict(ds_pca.attrs))

out["regime_labels"] = xr.DataArray(
    regime_labels.astype(np.int16),
    dims=[time_dim],
    coords={time_dim: time_coord},
)

out["centroids_pc"] = xr.DataArray(
    centroids.astype(np.float32),
    dims=["regime", mode_dim],
    coords={"regime": np.arange(1, FINAL_K + 1), mode_dim: PC_da[mode_dim].values},
)

out["frequency"] = xr.DataArray(
    frac.astype(np.float32),
    dims=["regime"],
    coords={"regime": np.arange(1, FINAL_K + 1)},
)

out["transition_counts"] = xr.DataArray(
    T.astype(np.int32),
    dims=["regime_from", "regime_to"],
    coords={
        "regime_from": np.arange(1, FINAL_K + 1),
        "regime_to": np.arange(1, FINAL_K + 1),
    },
)

out["transition_prob"] = xr.DataArray(
    P.astype(np.float32),
    dims=["regime_from", "regime_to"],
    coords={
        "regime_from": np.arange(1, FINAL_K + 1),
        "regime_to": np.arange(1, FINAL_K + 1),
    },
)

out["composite_weighted"] = xr.DataArray(
    composites.astype(np.float32),
    dims=["regime", lat_dim, lon_dim],
    coords={"regime": np.arange(1, FINAL_K + 1), lat_dim: lat, lon_dim: lon},
)

# Persistence summary as attributes (compact)
for r in range(1, FINAL_K + 1):
    out.attrs[f"persistence_mean_regime_{r}"] = float(pers[r]["mean"])
    out.attrs[f"persistence_median_regime_{r}"] = float(pers[r]["median"])
    out.attrs[f"persistence_n_runs_regime_{r}"] = int(pers[r]["n_runs"])

out.attrs.update(
    {
        "step": "09_final_clustering_regimes",
        "method": _METHOD_TAG,         # "ward" or "kmeans" (on-disk filename family)
        "cluster_mode": CLUSTER_MODE,   # "ward" or "ward_kmeans" (exact algorithm actually used)
        "final_k": int(FINAL_K),
        "random_state": int(RANDOM_STATE),
        "source_pca": str(paths["in_pca"]),
        "source_preproc": str(paths["in_preproc"]),
        "source_step8_hc": _ward_hc_path or "",
        "note": "Regime composites are computed from the actual weighted anomaly fields (X_matrix) averaged over days in each regime.",
    }
)

# Atomic write: write tmp then replace (avoids locked-file overwrite issues)
import os
from pathlib import Path

paths["out_regimes"].parent.mkdir(parents=True, exist_ok=True)
_target = Path(paths["out_regimes"])
_tmp = _target.with_suffix(_target.suffix + ".tmp")

try:
    out.to_netcdf(_tmp)
    os.replace(_tmp, _target)
    saved_path = _target
except PermissionError:
    # If overwrite is denied (file lock / sync), write a copy instead.
    if _tmp.exists():
        try:
            _tmp.unlink()
        except Exception:
            pass
    saved_path = _target.with_name(_target.stem + "_copy" + _target.suffix)
    out.to_netcdf(saved_path)

saved_path


## Evaluation of the final regime clustering (K-means)

This section adds **per-regime** quality diagnostics and writes them to the same NetCDF output file.

> Note: physical-space pattern correlation is often the easiest to explain in a paper, because it ties the cluster to the actual Z500 anomaly maps.


In [ ]:
# Setup (robust: can reload labels from saved NetCDF)
from pathlib import Path

import numpy as np
import xarray as xr
from sklearn.metrics import silhouette_samples, adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import pairwise_distances

# ---- Load regime labels / centroids if not in memory ----
if "regime_labels" not in globals():
    ds_reg = None

    # 1) If an output Dataset exists in memory, use it
    if "out" in globals() and isinstance(out, xr.Dataset) and ("regime_labels" in out):
        ds_reg = out

    # 2) Else try to open the saved NetCDF (preferred)
    elif "paths" in globals() and isinstance(paths, dict) and ("out_regimes" in paths):
        p = Path(paths["out_regimes"])  # may be PosixPath already
        if p.exists():
            # Open, read needed arrays, then close immediately to avoid file locks.
            with xr.open_dataset(p) as _ds:
                regime_labels = _ds["regime_labels"].values
                if "centroids" not in globals() and ("centroids_pc" in _ds):
                    centroids = _ds["centroids_pc"].values
            ds_reg = "closed"  # sentinel

    if ds_reg is None:
        raise NameError(
            "`regime_labels` is not defined. Run the clustering cells above first, "
            "or ensure `paths['out_regimes']` points to an existing NetCDF produced by this notebook."
        )

    if "regime_labels" not in globals():
        # when ds_reg came from `out` in memory
        regime_labels = ds_reg["regime_labels"].values
        if "centroids" not in globals() and ("centroids_pc" in ds_reg):
            centroids = ds_reg["centroids_pc"].values

# ---- Core inputs (must exist for the chosen metrics) ----
labels = np.asarray(regime_labels, dtype=int)  # 1..K

if "FINAL_K" in globals():
    K = int(FINAL_K)
else:
    K = int(np.nanmax(labels))

if "X_pca" not in globals():
    raise NameError("`X_pca` is not defined. Run the PCA/clustering prep cells above before evaluating.")

if "centroids" not in globals():
    raise NameError("`centroids` is not defined. Run the clustering cell above (Ward or K-means), or load a NetCDF that includes `centroids_pc`.")

centroids_pc = np.asarray(centroids, dtype=np.float64)
X_pc = np.asarray(X_pca, dtype=np.float64)



### 1) Cohesion (how tight is regime \(r\)?) — computed in **PC/EOF space**
K-means explicitly minimizes within-cluster squared distance in PC space, so cohesion is most consistent there.

- **Distance to centroid** (mean/median/p90/p95): lower means members of regime \(r\) are closer to its centroid → more compact regime.
  - **Interpretation**: these values are scale-dependent (depend on the number of PCs and their scaling), so use them mainly to compare regimes *within the same run*.
  - **Rule of thumb**: regimes with very large p95 (relative to others) often contain mixed states / outliers.

- **Silhouette (per regime)**: ranges from \(-1\) to \(1\). Higher is better.
  - **\(>0.5\)**: well separated / cohesive
  - **\(0.25–0.5\)**: some overlap but still useful
  - **\(<0.25\)**: substantial overlap
  - **negative**: many days are closer to another regime than to their assigned one




In [ ]:
# 1) Cohesion: distance-to-centroid summaries + per-regime silhouette (PC space)
dist_to_centroid = np.full(X_pc.shape[0], np.nan, dtype=np.float64)
for r in range(1, K + 1):
    idx = np.where(labels == r)[0]
    if idx.size == 0:
        continue
    c = centroids_pc[r - 1]
    dist_to_centroid[idx] = np.linalg.norm(X_pc[idx] - c[None, :], axis=1)

cohesion_mean = np.full(K, np.nan)
cohesion_median = np.full(K, np.nan)
cohesion_p90 = np.full(K, np.nan)
cohesion_p95 = np.full(K, np.nan)
cohesion_radius_typical = np.full(K, np.nan)  # robust within-regime radius

for r in range(1, K + 1):
    d = dist_to_centroid[labels == r]
    if d.size == 0:
        continue
    cohesion_mean[r - 1] = float(np.mean(d))
    cohesion_median[r - 1] = float(np.median(d))
    cohesion_p90[r - 1] = float(np.quantile(d, 0.90))
    cohesion_p95[r - 1] = float(np.quantile(d, 0.95))
    cohesion_radius_typical[r - 1] = float(np.median(d))

sil_samples = silhouette_samples(X_pc, labels, metric="euclidean")
sil_mean_by_regime = np.full(K, np.nan)
for r in range(1, K + 1):
    s = sil_samples[labels == r]
    if s.size == 0:
        continue
    sil_mean_by_regime[r - 1] = float(np.mean(s))

import pandas as pd
cohesion_table = pd.DataFrame(
    {
        "regime": np.arange(1, K + 1),
        "cohesion_dist_mean_pc": cohesion_mean,
        "cohesion_dist_median_pc": cohesion_median,
        "cohesion_dist_p90_pc": cohesion_p90,
        "cohesion_dist_p95_pc": cohesion_p95,
        "silhouette_mean_pc": sil_mean_by_regime,
    }
)
cohesion_table

### 2) Separation (how distinct is regime \(r\) from others?) — centroids in **PC space**
- **Nearest-centroid distance**: distance from centroid \(c_r\) to the closest other centroid. Higher means more distinct.
- **Separation-to-spread ratio**: \(\text{nearest-centroid distance} / \text{typical within-regime radius}\).
  - **\(> 1\)**: the regime’s centroid is farther from its nearest neighbor than its typical radius → usually distinct.
  - **\(< 1\)**: overlap likely (members can be as close to the neighbor as to their own centroid).




In [ ]:
# 2) Separation: nearest-centroid distance + separation/spread ratio (PC space)
Dcc = pairwise_distances(centroids_pc, metric="euclidean")
np.fill_diagonal(Dcc, np.inf)
nearest_centroid_dist = Dcc.min(axis=1).astype(np.float64)
separation_to_spread = nearest_centroid_dist / cohesion_radius_typical

separation_table = pd.DataFrame(
    {
        "regime": np.arange(1, K + 1),
        "nearest_centroid_dist_pc": nearest_centroid_dist,
        "separation_to_spread_pc": separation_to_spread,
    }
)
separation_table

### 3) Stability / robustness (is regime \(r\) reproducible?) — bootstrap resampling
We rerun K-means on bootstrap samples and compare the resulting full-sample assignments to the original.
Because cluster ids are arbitrary, we first **match bootstrap centroids to original centroids (Hungarian algorithm)**.

- **ARI (Adjusted Rand Index)**: 0 means random agreement, 1 means identical partitions.
- **NMI (Normalized Mutual Information)**: 0 to 1, higher is more agreement.
  - **\(\ge 0.8\)**: very stable
  - **\(0.6–0.8\)**: reasonably stable
  - **\(< 0.6\)**: sensitive to sampling (often means too many regimes, weak separation, or strong seasonality mixing)



In [ ]:
# 3) Stability / robustness: bootstrap resampling + Hungarian matching, then ARI/NMI
# Mode-aware: refits the same clustering method used for the official labels (CLUSTER_MODE) on each
# bootstrap resample, so the ARI/NMI comparison is apples-to-apples (Ward-vs-Ward or K-means-vs-K-means).
# If scipy is unavailable in the environment, this cell will fall back to NaNs.
try:
    from scipy.cluster.hierarchy import linkage as _linkage
    from scipy.optimize import linear_sum_assignment

    N_BOOT = 30
    rng = np.random.default_rng(RANDOM_STATE)
    ari_boot = np.empty(N_BOOT, dtype=np.float64)
    nmi_boot = np.empty(N_BOOT, dtype=np.float64)

    for b in range(N_BOOT):
        boot_idx = rng.integers(0, X_pc.shape[0], size=X_pc.shape[0])

        if CLUSTER_MODE == "ward":
            _Zb = _linkage(X_pc[boot_idx], method="ward")
            _asgn_b = fcluster(_Zb, K, criterion="maxclust")
            Cb = np.stack(
                [X_pc[boot_idx][_asgn_b == c].mean(axis=0) for c in range(1, K + 1)], axis=0
            )
            labels_b = np.argmin(pairwise_distances(X_pc, Cb), axis=1).astype(int) + 1
        else:
            km_b = KMeans(
                n_clusters=K,
                n_init=50,
                random_state=int(rng.integers(0, 1_000_000_000)),
            )
            km_b.fit(X_pc[boot_idx])
            labels_b = km_b.predict(X_pc).astype(int) + 1  # 1..K
            Cb = np.asarray(km_b.cluster_centers_, dtype=np.float64)

        # Match bootstrap regime ids to original regime ids by centroid distances
        cost = pairwise_distances(centroids_pc, Cb, metric="euclidean")
        row_ind, col_ind = linear_sum_assignment(cost)
        mapping = {int(col_ind[i] + 1): int(row_ind[i] + 1) for i in range(K)}
        labels_b_matched = np.vectorize(mapping.get)(labels_b)

        ari_boot[b] = adjusted_rand_score(labels, labels_b_matched)
        nmi_boot[b] = normalized_mutual_info_score(labels, labels_b_matched)

    stability_ari_mean = float(np.mean(ari_boot))
    stability_ari_std = float(np.std(ari_boot))
    stability_nmi_mean = float(np.mean(nmi_boot))
    stability_nmi_std = float(np.std(nmi_boot))
    stability_error = None

except Exception as e:
    N_BOOT = 30
    stability_ari_mean = np.nan
    stability_ari_std = np.nan
    stability_nmi_mean = np.nan
    stability_nmi_std = np.nan
    stability_error = str(e)

pd.DataFrame(
    {
        "metric": ["ARI", "NMI"],
        "mean": [stability_ari_mean, stability_nmi_mean],
        "std": [stability_ari_std, stability_nmi_std],
        "n_boot": [N_BOOT, N_BOOT],
    }
)


### 4) Physical-space “pattern purity” (do members look like the composite?)
Using the anomaly fields \(X\_\text{matrix}(time, space)\) and the regime composite maps:

- **Pattern correlation (per day vs its regime composite)**: reported as median and IQR (p25–p75) per regime.
  - **Interpretation** (very rough guidance):
    - **\(>0.6\)**: strong, consistent physical pattern
    - **\(0.3–0.6\)**: moderate consistency
    - **\(<0.3\)**: weak / mixed pattern

In [ ]:
# 4) Physical-space pattern purity: corr(day anomaly map, regime composite map)
# (Uses the same anomaly fields used to compute composites.)
if "X_maps" not in globals():
    n_lat, n_lon = len(lat), len(lon)
    X_maps = X_matrix.reshape((X_matrix.shape[0], n_lat, n_lon))

composites_maps = np.asarray(composites, dtype=np.float64)  # (K, lat, lon)

patterncorr_median = np.full(K, np.nan)
patterncorr_p25 = np.full(K, np.nan)
patterncorr_p75 = np.full(K, np.nan)

for r in range(1, K + 1):
    idx = np.where(labels == r)[0]
    if idx.size == 0:
        continue

    y = composites_maps[r - 1].ravel()
    y0 = y - y.mean()
    y_norm = np.linalg.norm(y0)
    if y_norm == 0:
        continue

    cors = np.empty(idx.size, dtype=np.float64)
    for j, t in enumerate(idx):
        x = np.asarray(X_maps[t], dtype=np.float64).ravel()
        x0 = x - x.mean()
        x_norm = np.linalg.norm(x0)
        cors[j] = np.nan if x_norm == 0 else float(np.dot(x0, y0) / (x_norm * y_norm))

    patterncorr_median[r - 1] = float(np.nanmedian(cors))
    patterncorr_p25[r - 1] = float(np.nanquantile(cors, 0.25))
    patterncorr_p75[r - 1] = float(np.nanquantile(cors, 0.75))

patterncorr_table = pd.DataFrame(
    {
        "regime": np.arange(1, K + 1),
        "patterncorr_median": patterncorr_median,
        "patterncorr_p25": patterncorr_p25,
        "patterncorr_p75": patterncorr_p75,
    }
)
patterncorr_table

In [ ]:
# Save evaluation metrics into the output Dataset and overwrite NetCDF
regime_coord = np.arange(1, K + 1)

out["cohesion_dist_mean_pc"] = xr.DataArray(
    cohesion_mean.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["cohesion_dist_median_pc"] = xr.DataArray(
    cohesion_median.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["cohesion_dist_p90_pc"] = xr.DataArray(
    cohesion_p90.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["cohesion_dist_p95_pc"] = xr.DataArray(
    cohesion_p95.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["silhouette_mean_pc"] = xr.DataArray(
    sil_mean_by_regime.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)

out["nearest_centroid_dist_pc"] = xr.DataArray(
    nearest_centroid_dist.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["separation_to_spread_pc"] = xr.DataArray(
    separation_to_spread.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)

out["patterncorr_median_composite"] = xr.DataArray(
    patterncorr_median.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["patterncorr_p25_composite"] = xr.DataArray(
    patterncorr_p25.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)
out["patterncorr_p75_composite"] = xr.DataArray(
    patterncorr_p75.astype(np.float32), dims=["regime"], coords={"regime": regime_coord}
)

out.attrs["stability_bootstrap_n"] = int(N_BOOT)
out.attrs["stability_bootstrap_ARI_mean"] = float(stability_ari_mean)
out.attrs["stability_bootstrap_ARI_std"] = float(stability_ari_std)
out.attrs["stability_bootstrap_NMI_mean"] = float(stability_nmi_mean)
out.attrs["stability_bootstrap_NMI_std"] = float(stability_nmi_std)
if stability_error is not None:
    out.attrs["stability_bootstrap_error"] = stability_error

# Atomic overwrite (same rationale as above)
import os
from pathlib import Path

_target = Path(paths["out_regimes"])
_tmp = _target.with_suffix(_target.suffix + ".tmp")

try:
    out.to_netcdf(_tmp)
    os.replace(_tmp, _target)
    saved_path = _target
except PermissionError:
    if _tmp.exists():
        try:
            _tmp.unlink()
        except Exception:
            pass
    saved_path = _target.with_name(_target.stem + "_copy" + _target.suffix)
    out.to_netcdf(saved_path)

print("Saved evaluation metrics to:", saved_path)
